# 🚦 Monitor de Radares Ativos — MG (INMETRO/RBMLQ)

Este notebook:
1. Busca radares verificados recentemente (últimos 7 dias) em MG
2. Geocodifica as coordenadas via **vGeo MG** com base no KM
3. Envia um **e-mail** com os dados encontrados
4. Salva uma **planilha completa** no **Google Drive**

> **Fonte:** https://servicos.rbmlq.gov.br/dados-abertos/mg/medidores.json
> **vGeo MG:** https://vgeo.mg.gov.br


## ⚙️ 1. Configurações — preencha antes de rodar

In [2]:
# ============================================================
#  CONFIGURAÇÕES DO USUÁRIO
# ============================================================

# --- E-mail ---
EMAIL_REMETENTE   = "andresoaresdiniz201218@gmail.com"   # Gmail usado para enviar
EMAIL_DESTINATARIO = "andresoaresdiniz201218@gmail.com"  # Quem recebe o alerta
# Use uma 'Senha de App' do Google (não a senha normal):
# Acesse: myaccount.google.com > Segurança > Senhas de App
EMAIL_SENHA_APP   = "xxxx xxxx xxxx xxxx"   # 16 caracteres, com espaços

# --- Filtros ---
DIAS_RECENTES     = 7      # Janela de dias para considerar 'recente'
SITUACOES_ATIVAS  = ["aprovado", "ativo", "apto"]  # Status considerados ativos

# --- Google Drive ---
NOME_PLANILHA     = "Radares_Ativos_MG"    # Nome do arquivo .xlsx no Drive
PASTA_DRIVE       = ""                     # Deixe vazio para salvar na raiz

# --- vGeo MG ---
# API de geocodificação por rodovia/KM do governo de MG
VGEO_BASE_URL     = "https://vgeo.mg.gov.br/arcgis/rest/services"
# Fallback: Nominatim (OpenStreetMap) — usado se vGeo não retornar resultado
USAR_NOMINATIM_FALLBACK = True

print("✅ Configurações carregadas.")

✅ Configurações carregadas.


## 📦 2. Instalação de dependências

In [1]:
!pip install -q requests openpyxl geopy pandas tqdm


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\andre\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 📂 3. Montar o Google Drive

In [3]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_BASE = "/content/drive/MyDrive"
if PASTA_DRIVE:
    DRIVE_OUTPUT = os.path.join(DRIVE_BASE, PASTA_DRIVE)
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)
else:
    DRIVE_OUTPUT = DRIVE_BASE

PLANILHA_PATH = os.path.join(DRIVE_OUTPUT, f"{NOME_PLANILHA}.xlsx")
print(f"📁 Planilha será salva em: {PLANILHA_PATH}")

ModuleNotFoundError: No module named 'google.colab'

## 🔍 4. Buscar e filtrar radares da API RBMLQ

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import re

URL_API = "https://servicos.rbmlq.gov.br/dados-abertos/mg/medidores.json"

print(f"⏳ Buscando dados em: {URL_API}")
resp = requests.get(URL_API, timeout=30)
resp.raise_for_status()
dados_brutos = resp.json()

# Normaliza estrutura — a API pode retornar lista ou dict com chave 'medidores'
if isinstance(dados_brutos, list):
    registros = dados_brutos
elif isinstance(dados_brutos, dict):
    # Tenta chaves comuns
    for chave in ["medidores", "data", "instrumentos", "registros", "results"]:
        if chave in dados_brutos:
            registros = dados_brutos[chave]
            break
    else:
        registros = list(dados_brutos.values())[0]

print(f"📊 Total de registros recebidos: {len(registros)}")

# Mostra as chaves do primeiro registro para diagnóstico
if registros:
    print(f"🔑 Campos disponíveis: {list(registros[0].keys())}")

In [ ]:
# ============================================================
#  MAPEAMENTO DE CAMPOS
#  Ajuste os nomes das chaves conforme o retorno real da API
# ============================================================

def obter_campo(registro, *candidatos, padrao=""):
    """Tenta múltiplos nomes de campo em ordem, retorna o primeiro que encontrar."""
    for c in candidatos:
        for k, v in registro.items():
            if k.lower().replace("_","").replace(" ","") == c.lower().replace("_","").replace(" ",""):
                return str(v) if v is not None else padrao
    return padrao

def parse_data(texto):
    """Tenta converter string de data em objeto datetime."""
    formatos = ["%d/%m/%Y", "%Y-%m-%d", "%d-%m-%Y", "%Y/%m/%d", "%d/%m/%y"]
    for fmt in formatos:
        try:
            return datetime.strptime(texto.strip(), fmt)
        except (ValueError, AttributeError):
            continue
    return None

def extrair_km(texto):
    """Extrai o número do KM de uma string como 'RODOVIA BR-040 KM 512,3'."""
    match = re.search(r'km\s*[:]?\s*([\d]+[.,]?[\d]*)', texto, re.IGNORECASE)
    if match:
        return float(match.group(1).replace(",", "."))
    return None

def extrair_rodovia(texto):
    """Extrai código da rodovia de strings como 'BR-040', 'MG-050'."""
    match = re.search(r'((?:BR|MG|ES|SP|RJ|GO|MS|RS|SC|PR|BA|CE|PE|PA|AM|MT|TO|PI|MA|RN|PB|AL|SE|RO|RR|AC|AP|DF)[\s-]?[\d]{2,3})', texto, re.IGNORECASE)
    if match:
        return match.group(1).upper().replace(" ", "-")
    return None


# Filtro por data e situação
corte_data = datetime.now() - timedelta(days=DIAS_RECENTES)
radares_recentes = []

for r in registros:
    situacao = obter_campo(r, "situacao", "status", "resultado", "situação").lower()
    data_str  = obter_campo(r, "dataverificacao", "data_verificacao", "dataverif",
                             "datainicio", "data", "verificacao")
    data_dt   = parse_data(data_str)

    ativo   = any(s in situacao for s in SITUACOES_ATIVAS)
    recente = (data_dt is not None) and (data_dt >= corte_data)

    if ativo and recente:
        local    = obter_campo(r, "local", "localizacao", "endereco", "logradouro")
        municipio = obter_campo(r, "municipio", "cidade", "localidade")
        rodovia_raw = obter_campo(r, "rodovia", "via", "br", local)

        radares_recentes.append({
            "numero":          obter_campo(r, "numero", "numeroserie", "serie", "id"),
            "municipio":       municipio,
            "local":           local,
            "rodovia":         extrair_rodovia(rodovia_raw + " " + local) or rodovia_raw,
            "km":              extrair_km(local),
            "situacao":        obter_campo(r, "situacao", "status"),
            "data_verificacao": data_str,
            "data_validade":   obter_campo(r, "datavalidade", "data_validade", "validade", "datafim"),
            "latitude":        None,
            "longitude":       None,
            "fonte_geo":       None,
            "maps_link":       None,
        })

print(f"\n✅ Radares ATIVOS nos últimos {DIAS_RECENTES} dias: {len(radares_recentes)}")

## 🌍 5. Geocodificação — vGeo MG + Nominatim (fallback)

In [ ]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
import time

nominatim = Nominatim(user_agent="radar_monitor_mg_colab")
geocode_nominatim = RateLimiter(nominatim.geocode, min_delay_seconds=1.1)


def geocode_vgeo(rodovia: str, km: float):
    """
    Tenta geocodificar via vGeo MG (ArcGIS REST).
    Serviço de referências lineares de rodovias de MG.
    """
    if not rodovia or km is None:
        return None, None
    try:
        # Endpoint de localização linear por rodovia e KM
        url = (
            f"{VGEO_BASE_URL}/Rodovias/Rodovias_MG/MapServer/exts/"
            f"LRSServer/networkLayers/0/geometryAtMeasure"
        )
        params = {
            "routeId":    rodovia.replace("-", ""),
            "measure":    km,
            "f":          "json",
        }
        r = requests.get(url, params=params, timeout=10)
        data = r.json()
        geom = data.get("geometryType") and data.get("geometry")
        if geom and "x" in geom:
            # vGeo retorna em SIRGAS 2000 (EPSG:4674) ≈ WGS84
            return round(geom["y"], 6), round(geom["x"], 6)

        # Tenta endpoint alternativo: Find Address
        url2 = f"{VGEO_BASE_URL}/Geocoding/Geocoder/GeocodeServer/findAddressCandidates"
        params2 = {
            "SingleLine": f"{rodovia} KM {km} Minas Gerais",
            "outFields":  "*",
            "f":          "json",
        }
        r2 = requests.get(url2, params=params2, timeout=10)
        data2 = r2.json()
        cands = data2.get("candidates", [])
        if cands:
            loc = cands[0]["location"]
            return round(loc["y"], 6), round(loc["x"], 6)
    except Exception:
        pass
    return None, None


def geocode_nominatim_func(municipio: str, rodovia: str, km: float):
    """Fallback via OpenStreetMap/Nominatim."""
    try:
        query = f"{rodovia} km {km}, {municipio}, Minas Gerais, Brasil"
        loc = geocode_nominatim(query, country_codes="br")
        if loc:
            return round(loc.latitude, 6), round(loc.longitude, 6)
        # Tenta só município
        loc2 = geocode_nominatim(f"{municipio}, Minas Gerais, Brasil", country_codes="br")
        if loc2:
            return round(loc2.latitude, 6), round(loc2.longitude, 6)
    except Exception:
        pass
    return None, None


print("🌍 Geocodificando radares...")
for radar in tqdm(radares_recentes):
    lat, lon = geocode_vgeo(radar["rodovia"], radar["km"])
    fonte = "vGeo MG"

    if (lat is None) and USAR_NOMINATIM_FALLBACK:
        lat, lon = geocode_nominatim_func(radar["municipio"], radar["rodovia"], radar["km"])
        fonte = "Nominatim (OSM)"

    radar["latitude"]  = lat
    radar["longitude"] = lon
    radar["fonte_geo"] = fonte if lat else "Não encontrado"
    if lat and lon:
        radar["maps_link"] = f"https://www.google.com/maps?q={lat},{lon}"

print("\n✅ Geocodificação concluída.")
geo_ok = sum(1 for r in radares_recentes if r["latitude"])
print(f"   📍 Com coordenadas: {geo_ok}/{len(radares_recentes)}")

## 📊 6. Visualizar tabela no Colab

In [ ]:
df = pd.DataFrame(radares_recentes)

if df.empty:
    print("ℹ️ Nenhum radar ativo encontrado nos últimos", DIAS_RECENTES, "dias.")
else:
    print(f"📋 {len(df)} radar(es) encontrado(s):")
    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_colwidth", 60)
    display(df)

## 💾 7. Salvar planilha no Google Drive

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

if not df.empty:
    wb = Workbook()
    ws = wb.active
    ws.title = "Radares Ativos"

    # Cabeçalho formatado
    COLUNAS = [
        "Número", "Município", "Local", "Rodovia", "KM",
        "Situação", "Data Verificação", "Data Validade",
        "Latitude", "Longitude", "Fonte Geo", "Google Maps"
    ]
    COR_HEADER = "1F4E79"
    COR_PAR    = "D6E4F0"

    header_fill   = PatternFill("solid", fgColor=COR_HEADER)
    par_fill      = PatternFill("solid", fgColor=COR_PAR)
    thin_border   = Border(
        left=Side(style="thin"), right=Side(style="thin"),
        top=Side(style="thin"),  bottom=Side(style="thin")
    )

    for col_idx, col_name in enumerate(COLUNAS, 1):
        cell = ws.cell(row=1, column=col_idx, value=col_name)
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.fill      = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border    = thin_border
    ws.row_dimensions[1].height = 22

    # Dados
    campos_df = ["numero","municipio","local","rodovia","km",
                 "situacao","data_verificacao","data_validade",
                 "latitude","longitude","fonte_geo","maps_link"]

    for row_idx, radar in enumerate(radares_recentes, 2):
        fill = par_fill if row_idx % 2 == 0 else None
        for col_idx, campo in enumerate(campos_df, 1):
            valor = radar.get(campo, "")
            cell  = ws.cell(row=row_idx, column=col_idx, value=valor)
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            cell.border    = thin_border
            if fill:
                cell.fill = fill
            # Link clicável para Maps
            if campo == "maps_link" and valor:
                cell.hyperlink = valor
                cell.value     = "📍 Abrir"
                cell.font      = Font(color="0563C1", underline="single")

    # Ajustar larguras
    larguras = [12, 20, 40, 12, 8, 12, 16, 14, 12, 12, 16, 12]
    for i, larg in enumerate(larguras, 1):
        ws.column_dimensions[get_column_letter(i)].width = larg

    # Aba de metadados
    ws_meta = wb.create_sheet("Info")
    ws_meta["A1"] = "Gerado em:"
    ws_meta["B1"] = datetime.now().strftime("%d/%m/%Y %H:%M:%S")
    ws_meta["A2"] = "Fonte:"
    ws_meta["B2"] = "https://servicos.rbmlq.gov.br/dados-abertos/mg/medidores.json"
    ws_meta["A3"] = "Janela analisada:"
    ws_meta["B3"] = f"Últimos {DIAS_RECENTES} dias"
    ws_meta["A4"] = "Total encontrado:"
    ws_meta["B4"] = len(radares_recentes)

    wb.save(PLANILHA_PATH)
    print(f"\n💾 Planilha salva com sucesso em:\n   {PLANILHA_PATH}")
else:
    print("ℹ️ Nenhum dado para salvar.")

## 📧 8. Enviar e-mail de alerta

In [ ]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text      import MIMEText
from email.mime.base      import MIMEBase
from email              import encoders

def montar_html(radares):
    """Gera o corpo HTML do e-mail com tabela estilizada."""
    hoje = datetime.now().strftime("%d/%m/%Y %H:%M")
    linhas_html = ""
    for i, r in enumerate(radares):
        cor_linha = "#EAF4FB" if i % 2 == 0 else "#FFFFFF"
        maps = f'<a href="{r["maps_link"]}">📍 Ver no Maps</a>' if r.get("maps_link") else "—"
        linhas_html += f"""
        <tr style="background:{cor_linha}">
          <td>{r.get('numero','—')}</td>
          <td>{r.get('municipio','—')}</td>
          <td>{r.get('rodovia','—')}</td>
          <td>{r.get('km','—')}</td>
          <td>{r.get('local','—')}</td>
          <td>{r.get('situacao','—')}</td>
          <td>{r.get('data_verificacao','—')}</td>
          <td>{r.get('data_validade','—')}</td>
          <td>{r.get('latitude','—')}</td>
          <td>{r.get('longitude','—')}</td>
          <td>{r.get('fonte_geo','—')}</td>
          <td>{maps}</td>
        </tr>"""

    return f"""
    <html><body style="font-family:Arial,sans-serif;color:#222;">
    <h2 style="color:#1F4E79;">🚦 Alerta — Radares Ativos MG (últimos {DIAS_RECENTES} dias)</h2>
    <p><b>Data/Hora:</b> {hoje}<br>
       <b>Total encontrado:</b> {len(radares)} radar(es)</p>
    <table border="1" cellspacing="0" cellpadding="6"
           style="border-collapse:collapse;font-size:13px;width:100%;">
      <thead>
        <tr style="background:#1F4E79;color:#fff;">
          <th>Número</th><th>Município</th><th>Rodovia</th><th>KM</th><th>Local</th>
          <th>Situação</th><th>Dt. Verif.</th><th>Validade</th>
          <th>Latitude</th><th>Longitude</th><th>Fonte Geo</th><th>Mapa</th>
        </tr>
      </thead>
      <tbody>{linhas_html}</tbody>
    </table>
    <br><p style="color:#777;font-size:12px;">
      Fonte: INMETRO/RBMLQ — Medidores de Velocidade MG<br>
      Planilha completa disponível no Google Drive.
    </p>
    </body></html>
    """


def enviar_email(radares):
    if not radares:
        print("ℹ️ Sem radares recentes — e-mail não enviado.")
        return

    msg = MIMEMultipart("mixed")
    msg["From"]    = EMAIL_REMETENTE
    msg["To"]      = EMAIL_DESTINATARIO
    msg["Subject"] = (
        f"🚦 [{len(radares)} radar(es)] Novos Medidores Ativos MG — "
        f"{datetime.now().strftime('%d/%m/%Y')}"
    )

    corpo = MIMEText(montar_html(radares), "html", "utf-8")
    msg.attach(corpo)

    # Anexar planilha se existir
    if os.path.exists(PLANILHA_PATH):
        with open(PLANILHA_PATH, "rb") as f:
            anexo = MIMEBase("application",
                             "vnd.openxmlformats-officedocument.spreadsheetml.sheet")
            anexo.set_payload(f.read())
        encoders.encode_base64(anexo)
        nome_arquivo = os.path.basename(PLANILHA_PATH)
        anexo.add_header("Content-Disposition", "attachment",
                         filename=nome_arquivo)
        msg.attach(anexo)
        print(f"📎 Planilha '{nome_arquivo}' será anexada ao e-mail.")

    # Envio via Gmail SMTP
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as servidor:
        servidor.login(EMAIL_REMETENTE, EMAIL_SENHA_APP)
        servidor.sendmail(EMAIL_REMETENTE, EMAIL_DESTINATARIO, msg.as_string())

    print(f"\n✅ E-mail enviado para {EMAIL_DESTINATARIO}")


enviar_email(radares_recentes)

## ⏰ 9. (Opcional) Agendar execução automática a cada N horas

In [ ]:
# Execute este bloco se quiser rodar o monitoramento em loop
# enquanto a sessão do Colab estiver ativa.
# O Colab encerra sessões ociosas (~90 min sem interação).

import time

INTERVALO_HORAS = 6   # Repetir a cada quantas horas
MAX_EXECUCOES   = 4   # Número máximo de repetições (0 = infinito)

RODAR_LOOP = False    # Mude para True para ativar

if RODAR_LOOP:
    execucao = 0
    while MAX_EXECUCOES == 0 or execucao < MAX_EXECUCOES:
        execucao += 1
        print(f"\n{'='*50}")
        print(f"🔄 Execução #{execucao} — {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
        print(f"{'='*50}")

        # Re-executa os blocos principais
        %run -i radar_monitor_mg.ipynb  # Alternativa: reexecutar células individualmente

        if MAX_EXECUCOES > 0 and execucao >= MAX_EXECUCOES:
            print("\n🏁 Limite de execuções atingido.")
            break

        proximo = datetime.now() + timedelta(hours=INTERVALO_HORAS)
        print(f"\n⏳ Próxima execução em {INTERVALO_HORAS}h — {proximo.strftime('%d/%m/%Y %H:%M')}")
        time.sleep(INTERVALO_HORAS * 3600)
else:
    print("ℹ️ Loop desativado. Mude RODAR_LOOP = True para ativar.")

---
## 📌 Referências
| Recurso | URL |
|---|---|
| API RBMLQ/MG | https://servicos.rbmlq.gov.br/dados-abertos/mg/medidores.json |
| Portal vGeo MG | https://vgeo.mg.gov.br |
| Metadados INMETRO | https://servicos.rbmlq.gov.br/dados-abertos/metadados_medidores_velocidade.pdf |
| Senha de App Gmail | https://myaccount.google.com/apppasswords |
